# LangGraph with AgentCore Memory Checkpointer (Short term memory)

# LangGraph 与 AgentCore Memory 检查点器（短期记忆）

## Introduction

## 简介

This notebook demonstrates how to integrate Amazon Bedrock AgentCore Memory capabilities with LangGraph using the **AgentCoreMemorySaver** checkpointer. We'll focus on **short-term memory** persistence across conversation turns - allowing an agent to maintain running context and build upon previous calculations through automatic state checkpointing.

本笔记本演示如何使用 **AgentCoreMemorySaver** 检查点器将 Amazon Bedrock AgentCore Memory 功能与 LangGraph 集成。我们将重点关注跨对话轮次的**短期记忆**持久化 - 允许代理维护运行上下文并通过自动状态检查点在先前计算的基础上继续构建。

## Tutorial Details

## 教程详情

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Short Term Conversational                                                        |
| Agent usecase       | Multi-step Math Calculations                                                     |
| Agentic Framework   | Langgraph                                                                        |
| LLM model           | Anthropic Claude Haiku 4.5                                                      |
| Tutorial components | AgentCore Short-term Memory, Langgraph Checkpointer, Math Tools                |
| Example complexity  | Beginner                                                                         |

| 信息                 | 详情                                                                              |
|:--------------------|:---------------------------------------------------------------------------------|
| 教程类型             | 短期对话                                                                          |
| 代理用例             | 多步数学计算                                                                       |
| 代理框架             | Langgraph                                                                        |
| LLM 模型            | Anthropic Claude Haiku 4.5                                                      |
| 教程组件             | AgentCore 短期记忆、Langgraph 检查点器、数学工具                                    |
| 示例复杂度           | 初级                                                                              |

You'll learn to:
- Create a memory checkpointer with AgentCore Memory for automatic state persistence
- Use LangGraph's built-in checkpointing system with AgentCore Memory backend
- Maintain conversation context across multiple interactions
- Inspect and manage conversation state and history

您将学习：
- 使用 AgentCore Memory 创建内存检查点器以实现自动状态持久化
- 使用带有 AgentCore Memory 后端的 LangGraph 内置检查点系统
- 在多次交互中维护对话上下文
- 检查和管理对话状态及历史

### Scenario Context

### 场景背景

In this example, we'll create a "**Math Agent**" that can perform multi-step mathematical calculations. Unlike simple one-off interactions, this agent uses AgentCore Memory's checkpointing capabilities to maintain running context, allowing it to build upon previous calculations and remember the conversation flow across multiple turns.

在本示例中，我们将创建一个"**数学代理**"，它可以执行多步数学计算。与简单的一次性交互不同，此代理使用 AgentCore Memory 的检查点功能来维护运行上下文，允许它在先前计算的基础上继续构建，并在多轮对话中记住对话流程。

## Architecture

## 架构

<div style="text-align:left">
    <img src="images/architecture.png" width="65%" />
</div>

## Prerequisites

## 前提条件

- Python 3.10+
- AWS account with appropriate permissions
- AWS IAM role with appropriate permissions for AgentCore Memory
- Access to Amazon Bedrock models

- Python 3.10+
- 具有适当权限的 AWS 账户
- 具有 AgentCore Memory 适当权限的 AWS IAM 角色
- 访问 Amazon Bedrock 模型

### How the Integration Works

### 集成工作原理

The integration between LangGraph and AgentCore Memory involves:

1. Using AgentCore Memory as a checkpointer backend for LangGraph state persistence
2. Automatic saving and loading of conversation state at each step
3. Support for multiple concurrent sessions and actors

LangGraph 与 AgentCore Memory 之间的集成包括：

1. 使用 AgentCore Memory 作为 LangGraph 状态持久化的检查点后端
2. 在每个步骤自动保存和加载对话状态
3. 支持多个并发会话和参与者

This approach provides seamless state management without requiring manual memory operations, creating a more maintainable and scalable agent architecture.

这种方法提供无缝的状态管理，无需手动内存操作，从而创建更易维护和可扩展的代理架构。

In [ ]:
# Install necessary libraries
!pip install -qr requirements.txt

In [ ]:
# Import LangGraph and LangChain components
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent

In [ ]:
# Import the AgentCoreMemorySaver that we will use as a checkpointer
import os
import logging

from langgraph_checkpoint_aws import AgentCoreMemorySaver
from bedrock_agentcore.memory import MemoryClient

region = os.getenv('AWS_REGION', 'us-west-2')
logging.getLogger("math-agent").setLevel(logging.DEBUG)

# Create or get the memory resource
memory_name = "MathLanggraphAgent"
client = MemoryClient(region_name=region)
memory = client.create_or_get_memory(name=memory_name)
memory_id = memory['id'] # Keep this memory ID for later use

### AgentCore Memory Configuration

### AgentCore Memory 配置

Now let's configure our AgentCore Memory checkpointer and initialize the LLM:

现在让我们配置 AgentCore Memory 检查点器并初始化 LLM：

- `memory_id` corresponds to our AgentCore Memory resource where checkpoints will be stored
- `region` specifies the AWS region for our resources
- `MODEL_ID` defines the Bedrock model that will power our LangGraph agent

- `memory_id` 对应于我们存储检查点的 AgentCore Memory 资源
- `region` 指定我们资源所在的 AWS 区域
- `MODEL_ID` 定义为我们的 LangGraph 代理提供支持的 Bedrock 模型

In [ ]:
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

# Initialize checkpointer for state persistence
checkpointer = AgentCoreMemorySaver(memory_id, region_name=region)

# Initialize LLM
llm = init_chat_model(MODEL_ID, model_provider="bedrock_converse", region_name=region)

### Mathematical Tools

### 数学工具

Let's define the mathematical tools our agent will use. For this demonstration, we'll provide two simple operations:

让我们定义代理将使用的数学工具。在本演示中，我们将提供两个简单的操作：

In [ ]:
@tool
def add(a: int, b: int):
    """Add two integers and return the result"""
    return a + b


@tool
def multiply(a: int, b: int):
    """Multiply two integers and return the result"""
    return a * b


tools = [add, multiply]

### LangGraph Agent Implementation

### LangGraph 代理实现

Now let's create our agent using LangGraph's `create_react_agent` builder with our AgentCore Memory checkpointer:

现在让我们使用 LangGraph 的 `create_react_agent` 构建器和我们的 AgentCore Memory 检查点器来创建代理：

In [ ]:
graph = create_react_agent(
    model=llm,
    tools=tools,
    prompt="You are a helpful assistant",
    checkpointer=checkpointer,
)

graph

## Step 4: Run the LangGraph Agent

## 步骤 4：运行 LangGraph 代理

We can now run the agent with our AgentCore Memory checkpointer integration.

现在我们可以使用 AgentCore Memory 检查点器集成来运行代理。

### Configuration Setup

### 配置设置

In LangGraph, config is a `RuntimeConfig` that contains attributes that are necessary at invocation time, for example user IDs or session IDs. You can read additional documentation here: [https://langchain-ai.github.io/langgraphjs/how-tos/configuration/](https://langchain-ai.github.io/langgraphjs/how-tos/configuration/)

在 LangGraph 中，config 是一个 `RuntimeConfig`，包含调用时所需的属性，例如用户 ID 或会话 ID。您可以在此处阅读更多文档：[https://langchain-ai.github.io/langgraphjs/how-tos/configuration/](https://langchain-ai.github.io/langgraphjs/how-tos/configuration/)

For the AgentCore Memory checkpointer (`AgentCoreMemorySaver`), we NEED to specify:
- `thread_id`: Maps to AgentCore session_id (unique conversation thread)
- `actor_id`: Maps to AgentCore actor_id (user, agent, or any other identifier)

对于 AgentCore Memory 检查点器（`AgentCoreMemorySaver`），我们需要指定：
- `thread_id`：映射到 AgentCore session_id（唯一的对话线程）
- `actor_id`：映射到 AgentCore actor_id（用户、代理或任何其他标识符）

In [ ]:
config = {
    "configurable": {
        "thread_id": "session-1", # REQUIRED: This maps to Bedrock AgentCore session_id under the hood
        "actor_id": "react-agent-1", # REQUIRED: This maps to Bedrock AgentCore actor_id under the hood
    }
}

inputs = {"messages": [{"role": "user", "content": "What is 1337 times 515321? Then add 412 and return the value to me."}]}

#### Congratulations! Your Agent is ready!!

#### 恭喜！您的代理已准备就绪！！

### Let's test the Agent

### 让我们测试代理

Let's run our first calculation to see the agent in action:

让我们运行第一个计算来看看代理的实际运行：

In [ ]:
for chunk in graph.stream(inputs, stream_mode="updates", config=config):
    print(chunk)

### Inspecting Agent State

### 检查代理状态

Let's examine the current conversation state stored in AgentCore Memory. The checkpointer automatically saves and retrieves state for our actor and session:

让我们检查存储在 AgentCore Memory 中的当前对话状态。检查点器会自动为我们的参与者和会话保存和检索状态：

In [ ]:
for message in graph.get_state(config).values.get("messages"):
    print(f"{message.type}: {message.text()}")
    print("=========================================")

### Viewing Checkpoint History

### 查看检查点历史

Let's explore the checkpoint history to see how the agent's state evolved during execution.  Checkpoints are listed in reverse chronological order (most recent appear first).

让我们探索检查点历史，查看代理状态在执行过程中是如何演变的。检查点按时间倒序排列（最近的排在最前面）。

In [ ]:
for checkpoint in graph.get_state_history(config):
    print(
        f"(Checkpoint ID: {checkpoint.config['configurable']['checkpoint_id']}) # of messages in state: {len(checkpoint.values.get('messages'))}"
    )

### Testing Memory Persistence

### 测试记忆持久化

Now let's test the power of our checkpointer by continuing the conversation. The agent should remember our previous calculations:

现在让我们通过继续对话来测试检查点器的强大功能。代理应该记住我们之前的计算：

In [ ]:
inputs = {"messages": [{"role": "user", "content": "What were the first calculations I asked you to do?"}]}

for chunk in graph.stream(inputs, stream_mode="updates", config=config):
    print(chunk)

### Starting a New Session

### 开始新会话

Let's demonstrate session isolation by creating a new conversation thread. The agent won't remember the previous calculations in this new session:

让我们通过创建一个新的对话线程来演示会话隔离。代理在这个新会话中不会记住之前的计算：

In [ ]:
config = {
    "configurable": {
        "thread_id": "session-2", # New session ID
        "actor_id": "react-agent-1", # Same Actor ID
    }
}

inputs = {"messages": [{"role": "user", "content": "What values did I ask you to multiply and add?"}]}
for chunk in graph.stream(inputs, stream_mode="updates", config=config):
    print(chunk)

## Summary

## 总结

In this notebook, we've demonstrated:

在本笔记本中，我们演示了：

1. How to create an AgentCore Memory resource for checkpointing
2. Building a LangGraph agent with automatic state persistence
3. Implementing mathematical tools for multi-step calculations
4. Using the AgentCoreMemorySaver as a checkpointer backend
5. Testing memory persistence and session isolation

1. 如何创建用于检查点的 AgentCore Memory 资源
2. 构建具有自动状态持久化功能的 LangGraph 代理
3. 实现用于多步计算的数学工具
4. 使用 AgentCoreMemorySaver 作为检查点后端
5. 测试记忆持久化和会话隔离

This integration showcases the power of combining LangGraph's structured workflows with AgentCore Memory's robust checkpointing capabilities to create stateful, persistent AI agents that can maintain context across multiple interactions.

这种集成展示了将 LangGraph 的结构化工作流与 AgentCore Memory 强大的检查点功能相结合的能力，以创建有状态、持久的 AI 代理，可以在多次交互中维护上下文。

The approach we've demonstrated can be extended to more complex use cases, including multi-agent systems, long-running workflows, and specialized state management based on conversation context.

我们演示的方法可以扩展到更复杂的用例，包括多代理系统、长时间运行的工作流以及基于对话上下文的专门状态管理。

### Clean up

### 清理

Let's delete the memory to clean up the resources used in this notebook.

让我们删除内存以清理本笔记本中使用的资源。

In [ ]:
#client.delete_memory_and_wait(memory_id = memory_id, max_wait = 300, poll_interval =10)